# 4. Exact and functional perturbation retrieval

Run from the repository root with the **MetacellFormer-env** Python 3.11 kernel. This example is independent of notebooks 01–03 and does not require MetaQ. Download only the K562 single-cell input, its fixed selected-cell metadata, the functional annotations, and the five model files:

If access is restricted, run `hf auth login` once before downloading.

```bash
python -m pip install huggingface_hub
hf download netbiolab/MetacellFormer-evaluation inputs/perturbation/K562.h5ad contracts/perturbation_retrieval/selected_cells.tsv contracts/perturbation_retrieval/functional_annotations.tsv --repo-type dataset --local-dir ../MetacellFormer-evaluation
hf download netbiolab/MetacellFormer config.json model.safetensors gene_vocabulary.json gene_reference_medians.npy training_resources/value_bin_edges.npy --local-dir ../MetacellFormer-model
```

The K562 input is about 2 GiB; download time is additional. These paths match the defaults below; set `METACELLFORMER_EVALUATION` and `METACELLFORMER_MODEL` for other locations. From the released selected-cell roster, the notebook uses batches 31 and 34; the first eight controls per batch and first eight perturbed cells per batch for each of `KRR1`, `RIOK1`, `RPS12`, `LSM2`, `LSM6`, and `PHAX` (112 single cells). It first embeds each cell, then computes target/batch deltas from those embeddings. **CPU** works (about 2 minutes on the tested two-thread CPU); change `device` to `cuda:0` for GPU embedding extraction.

Expected files under `example_outputs/perturbation_retrieval/`: `subset.h5ad` (112 × 8,563), `selected_cells.tsv` (112 rows), `embeddings.npz` (`embeddings` shape `(112, 1024)`, all `valid`), `results/delta_vectors.npy` `(12, 1024)`, and `results/{exact_perturbation_state_retrieval,functional_perturbation_retrieval}/query_results.tsv.gz` (12 rows each). Each task also writes `target_scores.tsv` (six rows). Completion means both retrieval tasks produce finite scores for all 12 queries. The scores from this small example are not the paper's full-roster results.

In [1]:
from pathlib import Path
import json
import os
import sys
import subprocess
import anndata as ad
import numpy as np
import pandas as pd
from benchmarks.shared._common import embeddings

repo = Path.cwd().resolve()
assert (repo / "benchmarks/tasks/perturbation_retrieval.py").is_file(), "Start Jupyter from the repository root"
data_root = Path(os.environ.get("METACELLFORMER_EVALUATION", "../MetacellFormer-evaluation")).expanduser().resolve()
model_dir = Path(os.environ.get("METACELLFORMER_MODEL", "../MetacellFormer-model")).expanduser().resolve()
source = data_root / "inputs/perturbation/K562.h5ad"
metadata = data_root / "contracts/perturbation_retrieval/selected_cells.tsv"
annotations = data_root / "contracts/perturbation_retrieval/functional_annotations.tsv"
assert all(path.is_file() for path in (source, metadata, annotations)), "Download the three evaluation files listed above"
assert all((model_dir / name).is_file() for name in (
    "config.json", "model.safetensors", "gene_vocabulary.json",
    "gene_reference_medians.npy", "training_resources/value_bin_edges.npy")), "Download the five model files listed above"
work = repo / "example_outputs/perturbation_retrieval"
work.mkdir(parents=True, exist_ok=True)
config = work / "runtime.json"
config.write_text(json.dumps({"MetacellFormer": {"checkpoint": str(model_dir)}}, indent=2) + "\n")
device = "cpu"  # Change to cuda:0 if available.

full = pd.read_csv(metadata, sep="\t")
targets = ["KRR1", "RIOK1", "RPS12", "LSM2", "LSM6", "PHAX"]
batches = [31, 34]
control = full[full.cell_role.eq("control") & full.batch_group.isin(batches)].groupby("batch_group", sort=True).head(8)
perturbed = full[full.cell_role.eq("perturbed") & full.batch_group.isin(batches) & full.target_gene.isin(targets)].groupby(["batch_group", "target_gene"], sort=True).head(8)
selected = pd.concat([control, perturbed]).sort_index()
assert len(selected) == 112 and selected.cell_id.is_unique
assert control.groupby("batch_group").size().eq(8).all() and len(control) == 16
assert perturbed.groupby(["batch_group", "target_gene"]).size().eq(8).all() and len(perturbed) == 96
functional = pd.read_csv(annotations, sep="\t").set_index("target_gene")
assert functional.loc[targets, "primary_clean_eligible"].eq(1).all()
selected.to_csv(work / "selected_cells.tsv", sep="\t", index=False)
print("Selected", len(selected), "single cells from", len(targets), "targets and", len(batches), "batches")

Selected 112 single cells from 6 targets and 2 batches


In [2]:
backed = ad.read_h5ad(source, backed="r")
try:
    cell_ids = selected.cell_id.astype(str).to_numpy()
    positions = backed.obs_names.get_indexer(cell_ids)
    assert (positions >= 0).all()
    subset = backed[positions].to_memory()
    assert subset.obs_names.astype(str).tolist() == cell_ids.tolist()
    subset.write_h5ad(work / "subset.h5ad")
finally:
    backed.file.close()
print("subset.h5ad", subset.shape)

subset.h5ad (112, 8563)


In [3]:
subprocess.run([sys.executable, "-m", "benchmarks.embeddings.extract_embeddings",
                "--input", str(work / "subset.h5ad"),
                "--output", str(work / "embeddings.npz"),
                "--config", str(config), "--model", "MetacellFormer",
                "--task", "perturbation", "--device", device, "--batch-size", "4"],
               cwd=repo, check=True)
values, returned, valid = embeddings(work / "embeddings.npz", cell_ids)
assert values.shape == (112, 1024) and valid.all() and np.isfinite(values).all()
print("embeddings.npz", values.shape, "valid", int(valid.sum()))

embeddings.npz (112, 1024) valid 112


In [4]:
subprocess.run([sys.executable, "-m", "benchmarks.tasks.perturbation_retrieval",
                "--embedding", str(work / "embeddings.npz"),
                "--metadata", str(work / "selected_cells.tsv"),
                "--annotations", str(annotations), "--model", "MetacellFormer",
                "--device", "cpu", "--output-dir", str(work / "results")],
               cwd=repo, check=True)
delta = np.load(work / "results/delta_vectors.npy")
assert delta.shape == (12, 1024) and np.isfinite(delta).all()
print("delta_vectors.npy", delta.shape)
for task in ("exact_perturbation_state_retrieval", "functional_perturbation_retrieval"):
    task_dir = work / "results" / task
    queries = pd.read_csv(task_dir / "query_results.tsv.gz", sep="\t")
    scores = pd.read_csv(task_dir / "target_scores.tsv", sep="\t")
    assert len(queries) == 12 and len(scores) == 6 and np.isfinite(queries["map"]).all()
    print(task, "queries", len(queries), "targets", len(scores), "example mean AP", round(queries["map"].mean(), 6))

delta_vectors.npy (12, 1024)
exact_perturbation_state_retrieval queries 12 targets 6 example mean AP 0.586111
functional_perturbation_retrieval queries 12 targets 6 example mean AP 0.965278
